# r-doc-mcp — API smoke test

Starts the FastAPI app on a free port and exercises `/health` and `/search`.

Run top to bottom. The last cell shuts the API down.
Needs an ingested `.chroma_db` — see the README's Ingest section.

In [ ]:
import socket, subprocess, sys, tempfile, time
from pathlib import Path

import httpx

# a live server keeps its old code and its old .env forever, so re-running this cell must
# replace the previous one rather than leak it and leave you querying a stale process
try:
    api.terminate()
    api.wait(timeout=10)
    print("replaced the previously started API")
except NameError:
    pass

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
with socket.socket() as s:  # let the OS pick a free port so an already-running API doesn't clash
    s.bind(("127.0.0.1", 0))
    PORT = s.getsockname()[1]
API_URL = f"http://127.0.0.1:{PORT}"

# a file, not a PIPE: nothing drains a pipe here, and uvicorn deadlocks once its buffer fills
LOG = Path(tempfile.gettempdir()) / f"r_api_{PORT}.log"
api = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "r_doc_mcp.api:app", "--port", str(PORT)],
    cwd=ROOT, stdout=LOG.open("w"), stderr=subprocess.STDOUT,
)

for _ in range(120):  # first request loads the embedding + chroma clients, so be patient
    if api.poll() is not None:
        raise RuntimeError(f"API died:\n{LOG.read_text()}")
    try:
        httpx.get(f"{API_URL}/health", timeout=2).raise_for_status()
        break
    except Exception:
        time.sleep(1)
else:
    api.terminate()
    raise RuntimeError(f"API never became healthy:\n{LOG.read_text()}")

print("API up at", API_URL, "| server log:", LOG)

In [ ]:
# 120s: a cold ollama loads the embedding model on the first query
client = httpx.Client(base_url=API_URL, timeout=120)

health = client.get("/health").json()
print(health)
assert health["documents"] > 0, "empty collection — ingest first"

In [ ]:
hits = client.post("/search", json={"query": "PIC-SURE", "k": 3}).json()

for h in hits:
    print(f"{h['score']:.3f}  {h['metadata'].get('source')}  {h['content'][:120]!r}")
assert hits, "search returned nothing"

In [ ]:
# doc_type filter — types come from the ingest, e.g. page, faq, event, fellow, update, video, docs.
# event/video chunks also carry a sortable `datetime` (event date / YouTube upload date).
filtered = client.post("/search", json={"query": "community hours", "k": 3, "doc_type": "event"}).json()
for h in filtered:
    m = h["metadata"]
    print(m.get("doc_type"), m.get("datetime"), "|", m.get("title", m.get("source")))
assert all(h["metadata"]["doc_type"] == "event" for h in filtered)

In [ ]:
# mode="keyword" — fuzzy literal word matching instead of embeddings.
# Ignores case/punctuation and tolerates small typos; score = occurrence count
# (higher = better), unlike embedding mode's distance (lower = better).
for q in ["PIC-SURE", "picsure", "picsur"]:  # exact, unpunctuated, typo — all should hit
    hits = client.post("/search", json={"query": q, "k": 3, "mode": "keyword"}).json()
    print(f"{q!r}:")
    for h in hits:
        print(f"  {h['score']:>4.0f}  {h['metadata'].get('source')}  {h['content'][:100]!r}")
    assert hits, f"keyword search for {q!r} returned nothing"
    assert hits[0]["score"] >= hits[-1]["score"]  # ranked by match count, descending

In [ ]:
# date range — results rank by relevance, NOT date, so bound the range with
# date_from/date_to and compare the returned dates yourself.
import datetime
today = datetime.date.today().isoformat()

# latest (most recent past) event: everything up to today, then take the max date
past = client.post("/search", json={"query": "BDC event", "k": 20, "doc_type": "event", "date_to": today}).json()
assert past, "no past events — ingest events first"
latest = max(past, key=lambda h: h["metadata"].get("date_num", 0))
m = latest["metadata"]
print("latest past event:", m.get("date"), "|", m.get("title", m.get("source")))

# future events: everything from today on, soonest first
future = client.post("/search", json={"query": "BDC event", "k": 20, "doc_type": "event", "date_from": today}).json()
if not future:
    print("no future events in the DB")
for h in sorted(future, key=lambda x: x["metadata"].get("date_num", 0))[:5]:
    m = h["metadata"]
    print("upcoming:", m.get("date"), "|", m.get("title", m.get("source")))

In [ ]:
try:
    api.terminate()
    api.wait(timeout=10)
    print("API stopped. Server log:\n")
    print(LOG.read_text())
except NameError:
    print("no API to stop — the boot cell hasn't run in this kernel")